In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda

# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
center = snap.Cat.Group['GroupPos'][0]
widths = np.array([500., 500., 500.], dtype=float)

# m_filter = 1000*snap.mass
# filter_length = (np.cbrt(3*m_filter/(4*np.pi*snap['0_Density']))).arepo
filter_length = 2*snap['0_Diameters']

Attempting to get derived variable: 0_Diameters...
	So we need the variable: 0_Volume...	[DONE]



In [ ]:
@cuda.jit(lineinfo=True)
def deposit_on_grid(pos, hsml, tile_widths,
                 variable, weights, offsets, npixs, center, widths, deposited_var, 
                 scratch, kernel_type):
    """
    scratch has same dimensions as deposited_var (npixs[0], npixs[1], npixs[2])
    """
    # threadindex
    ip = cuda.grid(1)

    # particle position
    xp = pos[ip, 0]
    yp = pos[ip, 1]
    zp = pos[ip, 2]

    # particle diameter (hsml)
    hsmlPart = hsml[ip]

    xmin = center[0] - widths[0] / 2 - hsmlPart
    xmax = center[0] + widths[0] / 2 + hsmlPart

    ymin = center[1] - widths[1] / 2 - hsmlPart
    ymax = center[1] + widths[1] / 2 + hsmlPart

    zmin = center[2] - widths[2] / 2 - hsmlPart
    zmax = center[2] + widths[2] / 2 + hsmlPart

    

    sidelength_x, sidelength_y, sidelength_z = widths
    nx, ny, nz = npixs

    # Check if this cell/particle is inside domain
    inside_domain = False
    if (xp > xmin) and (xp < xmax):
        if (yp > ymin) and (yp < ymax):
            if (zp > zmin) and (zp < zmax):
                inside_domain = True

    if inside_domain:

        # can be negative, i.e. the particle is outside interpolating region
        # but contributes to the deposition
        ip_tile_x = int( (xp - offsets[0]) // tile_widths[0])
        ip_tile_y = int( (yp - offsets[1]) // tile_widths[1])
        ip_tile_z = int( (zp - offsets[2]) // tile_widths[2])

        # relative coordinates w.r.t. center of tile
        delta_x = xp - offsets[0] - (ip_tile_x + 0.5) * tile_widths[0] 
        delta_y = yp - offsets[1] - (ip_tile_y + 0.5) * tile_widths[1] 
        delta_z = zp - offsets[2] - (ip_tile_z + 0.5) * tile_widths[2] 

        weight = 0.0
        weight_tmp = 0.0

        ip_tile_x_min = ip_tile_x - (- delta_x + \
                        hsmlPart + tile_widths[0] / 2) // tile_widths[0] 
        ip_tile_x_max = ip_tile_x + (delta_x +   \
                        hsmlPart + tile_widths[0] / 2) // tile_widths[0] 
    
        ip_tile_y_min = ip_tile_y - (- delta_y + \
                        hsmlPart + tile_widths[1] / 2) // tile_widths[1] 
        ip_tile_y_max = ip_tile_y + (delta_y +   \
                        hsmlPart + tile_widths[1] / 2) // tile_widths[1] 
    
        ip_tile_z_min = ip_tile_z - (- delta_z + \
                        hsmlPart + tile_widths[2] / 2) // tile_widths[2] 
        ip_tile_z_max = ip_tile_z + (delta_z +   \
                        hsmlPart + tile_widths[2] / 2) // tile_widths[2]
        
        # filter_type == 0 is nearest-grid-point (NGP)
        if filter_type == 0:
            # these are the indices of the closest grid-point
            # if particle is in tile i, its xcoord is \in [x_i, x_(i+1))
            # if delta_x > 0 it means that particle is closer to
            # x_(i+1) than to x_i
            xgrid_idx = ip_tile_x
            if (delta_x > 0): xgrid_idx += 1
            ygrid_idx = ip_tile_y
            if (delta_y > 0): ygrid_idx += 1
            zgrid_idx = ip_tile_z
            if (delta_z > 0): zgrid_idx += 1

            # there could be out-of-bound problems
            # need to select only those that are on the grid
            if (xgrid_idx >= 0) and (xgrid_idx < nx):
                if (ygrid_idx >= 0) and (ygrid_idx < ny):
                    if (zgrid_idx >= 0) and (zgrid_idx < nz):

                        weight_tmp = weights[ip]
                        # the following two need to be atomic add
                        cuda.atomic.add(scratch, (xgrid_idx, ygrid_idx, zgrid_idx), weight_tmp)
                        cuda.atomic.add(deposited_var, (xgrid_idx, ygrid_idx, zgrid_idx), variable[ip] * weight_tmp)

            
            
        # filter_type == 1 is cloud-in-cell (CIC), particle is a cube
        elif filter_type == 1:
            # these are the indices of the potentially overlapping grid-points
            # if particle is in tile i, its xcoord is \in [x_i, x_(i+1)) 
            # if with its hslm it overlaps tiles from ip_tile_x_min to 
            # ip_tile_x_max (inclusive), the grid indices that it can overlap with
            # go from ip_tile_x_min + 1 to ip_tile_x_max (inclusive)
            # e.g. if particle in tile 2 overlaps with tiles 1 and 3
            # the grid indices go from 2 to 3            
            for tile_x in range(ip_tile_x_min + 1, ip_tile_x_max + 1):
                for tile_y in range(ip_tile_y_min + 1, ip_tile_y_max + 1):
                    for tile_z in range(ip_tile_z_min + 1, ip_tile_z_max + 1):

                        xgrid_idx = tile_x
                        ygrid_idx = tile_y
                        zgrid_idx = tile_z

                        # there could be out-of-bound problems
                        # need to select only those that are on the grid
                        if (xgrid_idx >= 0) and (xgrid_idx < nx):
                            if (ygrid_idx >= 0) and (ygrid_idx < ny):
                                if (zgrid_idx >= 0) and (zgrid_idx < nz):

                                    # write CIC deposition routine
                                    for ip_other in range(start_index, start_index + n_particles):
                                        dist = distance(pos[ip], pos[ip_other])
                                        if dist < filter_length:
                                            weight_tmp = gaussian_kernel(dist, filter_length / filter_window) * weights[ip_other]
                                            weight += weight_tmp
                                            smooth_var[ip] += variable[ip_other] * weight_tmp
                                            hitsNeighbours[ip] += 1
        if weight > 0.:
            smooth_var[ip] /= weight


In [ ]:
class Lorenzo(tc.DepositCartesianGrid):
    def 

In [2]:
depo = Lorenzo(snap, center, widths,npix=128, threadsperblock=256, regionType='cartesian')